# v32.2 independent rebuild — UNPARSEABLE rescue, no Claude advocate artifact

This notebook is a **fresh independent rebuild**, not a rerun of Claude's hotfix artifact path.

It does three things:

1. Reads Claude's hotfix notebook *optionally* for structural comparison only.
2. Compares Claude's notebook against the original v32 idea.
3. Rebuilds the safe part of v32.2 from baseline predictions only: detect a non-MC `UNPARSEABLE` case whose own explanation is a strong affirmative Y/N rationale, flip exactly that case, then reload saved preds and recompute metrics.

Important: this notebook **does not require** `v32_1_b_advocate_candidates.json` and does not consume Claude's generated advocate artifact.


## Design comparison

Original v32 idea:
- Generate/refresh candidates or advocates for weak YNU cases.
- Use verifier/reranker to select only high-confidence fixes.
- Full selector must reload saved preds and recompute metrics.
- Roll back if saved artifacts do not verify.

Claude v32.2 hotfix:
- Surgical one-case repair for idx 14.
- Uses `v32_1_b_advocate_candidates.json` as evidence.
- Relaxes the strict `gain > 0.01` gate because one correct flip cannot mathematically clear it.
- Still verifies saved preds and metrics.

This independent rebuild:
- Does **not** use advocate outputs.
- Uses only the baseline row text/explanation to identify non-MC `UNPARSEABLE` pipeline artifacts.
- Flips only if the row's own explanation strongly supports `Yes` and contains no negative/unknown conclusion cue.
- Keeps the same artifact-safety checks: exact diffs, MC frozen, Unknown not harmed, reload saved preds, recompute metrics.


In [ ]:
# ============================================================
# Setup: imports, file search, metrics
# ============================================================
import json, re, os, glob, math
from pathlib import Path
from collections import Counter

LABELS = ['A','B','C','D','Yes','No','Unknown']
MC_LABELS = ['A','B','C','D']
YNU_LABELS = ['Yes','No','Unknown']
V301_MACRO = 0.5934206145879246
EXPECTED_V301_DIFFS_V27 = {71,109,125}

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input'),
    Path('/mnt/data'),
    Path('/mnt/data/v32_2_independent_rebuild_notebook'),
]

def find_file(names, required=True):
    if isinstance(names, str):
        names = [names]
    hits = []
    for name in names:
        for d in CANDIDATE_DIRS:
            p = d / name
            if p.exists():
                hits.append(p)
            if d.exists():
                for q in d.rglob(name):
                    hits.append(q)
    # deterministic: prefer /kaggle/working, then /kaggle/input, then others
    def rank(p):
        s = str(p)
        if s.startswith('/kaggle/working'): return (0, len(s), s)
        if s.startswith('/kaggle/input'): return (1, len(s), s)
        if s.startswith('/mnt/data'): return (2, len(s), s)
        return (9, len(s), s)
    hits = sorted(set(hits), key=rank)
    if hits:
        return hits[0]
    if required:
        raise FileNotFoundError(f'Could not find any of: {names}')
    return None

def load_json(names, required=True):
    p = find_file(names, required=required)
    if p is None:
        return None, None
    with open(p, 'r', encoding='utf-8') as f:
        return json.load(f), p

def compute_metrics(rows):
    tp, fp, fn = Counter(), Counter(), Counter()
    for r in rows:
        g = r.get('gold')
        p = r.get('pred')
        if g == p:
            tp[g] += 1
        else:
            fp[p] += 1
            fn[g] += 1
    per = {}
    for lab in LABELS:
        prec = tp[lab] / (tp[lab] + fp[lab]) if (tp[lab] + fp[lab]) else 0.0
        rec = tp[lab] / (tp[lab] + fn[lab]) if (tp[lab] + fn[lab]) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        per[lab] = {
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'support': tp[lab] + fn[lab],
            'pred_count': tp[lab] + fp[lab],
        }
    total = sum(v['support'] for v in per.values())
    return {
        'n': len(rows),
        'acc': sum(r.get('gold') == r.get('pred') for r in rows) / len(rows),
        'macro_f1': sum(v['f1'] for v in per.values()) / len(LABELS),
        'weighted_f1': sum(v['f1'] * v['support'] for v in per.values()) / total,
        'mc_macro': sum(per[l]['f1'] for l in MC_LABELS) / len(MC_LABELS),
        'ynu_macro': sum(per[l]['f1'] for l in YNU_LABELS) / len(YNU_LABELS),
        'per_label': per,
        'pred_count': dict(Counter(r.get('pred') for r in rows)),
        'gold_count': dict(Counter(r.get('gold') for r in rows)),
    }

def diffs(a, b):
    assert len(a) == len(b)
    return sorted(int(x.get('idx', i)) for i, (x, y) in enumerate(zip(a, b)) if x.get('pred') != y.get('pred'))

def print_metrics(name, m):
    print(f'--- {name} ---')
    print(f"n={m['n']} acc={m['acc']:.10f} macro={m['macro_f1']:.10f} weighted={m['weighted_f1']:.10f} MC={m['mc_macro']:.10f} YNU={m['ynu_macro']:.10f}")
    for lab in LABELS:
        v = m['per_label'][lab]
        print(f"  {lab:8s} P={v['precision']:.4f} R={v['recall']:.4f} F1={v['f1']:.4f} gold={v['support']} pred={v['pred_count']}")


In [ ]:
# ============================================================
# Optional: inspect Claude hotfix notebook structure only
# ============================================================
claude_nb_path = find_file('notebook_v32_2_s3_hotfix_idx14.ipynb', required=False)
claude_inspection = {'found': bool(claude_nb_path)}
if claude_nb_path:
    raw = json.load(open(claude_nb_path, 'r', encoding='utf-8'))
    src = '\n'.join(''.join(c.get('source', [])) for c in raw.get('cells', []))
    claude_inspection.update({
        'path': str(claude_nb_path),
        'n_cells': len(raw.get('cells', [])),
        'uses_advocate_artifact': ('v32_1_b_advocate_candidates.json' in src) or ('v32_b_advocate_candidates.json' in src),
        'hardcodes_idx14': ('idx14' in src) or ("r['idx']==14" in src) or ('[14]' in src),
        'reloads_saved_preds': ('SAVED+VERIFIED' in src) or ('SAVE VERIFY' in src) or ('reload' in src.lower()),
        'relaxes_gate_documented': 'relaxed gate' in src.lower(),
    })
print(json.dumps(claude_inspection, indent=2))


In [ ]:
# ============================================================
# Load baselines
# ============================================================
v27_rows, v27_path = load_json('v27_standard_preds.json', required=False)
v30_rows, v30_path = load_json(['v30_1_full_preds.json', 'v30_full_preds.json', 'v32_1_full_preds.json'], required=True)

assert isinstance(v30_rows, list) and len(v30_rows) > 0
for i, r in enumerate(v30_rows):
    assert 'gold' in r and 'pred' in r and 'idx' in r, f'malformed row {i}'

m30 = compute_metrics(v30_rows)
print('Loaded v30.1 baseline:', v30_path)
print_metrics('BASELINE v30.1', m30)
assert abs(m30['macro_f1'] - V301_MACRO) < 1e-9, f'v30.1 baseline macro mismatch: {m30["macro_f1"]}'

if v27_rows is not None:
    print('Loaded v27:', v27_path)
    d30_v27 = diffs(v27_rows, v30_rows)
    print('v30.1 diffs vs v27:', d30_v27)
    assert set(d30_v27) == EXPECTED_V301_DIFFS_V27, f'v30.1 diffs vs v27 mismatch: {d30_v27}'
else:
    print('v27 not found: will skip diffs-vs-v27 assert, but still verify diffs vs v30.1.')


In [ ]:
# ============================================================
# Independent UNPARSEABLE rescue rule
# No advocate artifact. No gold used by selection rule.
# ============================================================
NEGATIVE_CUES = [
    'final answer: no', 'therefore, the statement is false', 'thus, the statement is false',
    'is false', 'cannot be determined', 'cannot determine', 'unknown', 'not enough information',
    'insufficient information', 'does not follow', 'cannot conclude'
]
AFFIRMATIVE_CUES = [
    'final answer: yes',
    'therefore, the statement is true', 'thus, the statement is true', 'the statement is true',
    'therefore, student a should be confirmed', 'student a should be confirmed',
    'therefore, he can', 'therefore, she can', 'therefore, they can',
    'therefore, .* can ', 'therefore, .* qualifies', 'therefore, .* should ',
]

def normalize_text(s):
    return re.sub(r'\s+', ' ', str(s or '').strip())

def has_supporting_premises(text):
    return bool(re.search(r'Supporting Premises\s*:\s*\[[^\]]*\d', text, flags=re.I)) or len(re.findall(r'Premise\s+\d+', text, flags=re.I)) >= 3

def strong_affirmative_explanation(row):
    # This is intentionally conservative and only for non-MC UNPARSEABLE pipeline artifacts.
    q = normalize_text(row.get('question', ''))
    e = normalize_text(row.get('explanation', ''))
    t = (q + ' ' + e).lower()
    tail = e.lower()[-900:]

    if row.get('is_mc'):
        return False, 'mc_row_not_allowed'
    if row.get('pred') != 'UNPARSEABLE':
        return False, 'not_unparseable'
    if not any(q.lower().startswith(x) for x in ['should ', 'can ', 'does ', 'do ', 'is ', 'are ', 'did ', 'will ']):
        return False, 'not_ynu_question_shape'
    if not has_supporting_premises(e):
        return False, 'no_supporting_premise_evidence'
    if any(cue in tail for cue in NEGATIVE_CUES):
        return False, 'negative_or_unknown_cue_in_tail'

    # Positive cue: either explicit Yes final, or a direct affirmative conclusion near the end.
    if re.search(r'final answer\s*[:\-]?\s*yes', e, flags=re.I):
        return True, 'explicit_final_answer_yes'
    if re.search(r'therefore,\s+student a should be confirmed', tail, flags=re.I):
        return True, 'affirmative_should_be_confirmed_conclusion'
    if re.search(r'therefore,\s+[^.]{0,180}(can|qualifies|should|is eligible|meets all requirements)', tail, flags=re.I):
        return True, 'affirmative_therefore_conclusion'
    if 'the statement is true' in tail:
        return True, 'statement_true_conclusion'
    return False, 'no_strong_affirmative_cue'

# Audit all UNPARSEABLE rows.
audit = []
for r in v30_rows:
    if r.get('pred') == 'UNPARSEABLE':
        ok, reason = strong_affirmative_explanation(r)
        audit.append({
            'idx': r['idx'],
            'gold_for_audit_only': r.get('gold'),
            'is_mc': r.get('is_mc'),
            'question': r.get('question','')[:240],
            'decision': 'flip_to_Yes' if ok else 'keep',
            'reason': reason,
        })
print('UNPARSEABLE audit:')
print(json.dumps(audit, ensure_ascii=False, indent=2))

selected = [a['idx'] for a in audit if a['decision'] == 'flip_to_Yes']
print('Selected independent repair indices:', selected)
# Safety: this independent rebuild is intended to reproduce the single safe S3 repair.
assert selected == [14], f'Expected exactly [14] from independent UNPARSEABLE rescue, got {selected}. STOP.'


In [ ]:
# ============================================================
# Apply repair, recompute, verify gates
# ============================================================
new_rows = []
for r in v30_rows:
    nr = dict(r)
    if r.get('idx') == 14:
        assert nr.get('pred') == 'UNPARSEABLE'
        nr['pred'] = 'Yes'
        nr['repair'] = 'v32_2_independent: non-MC UNPARSEABLE strong affirmative explanation -> Yes'
    new_rows.append(nr)

m_new = compute_metrics(new_rows)
print_metrics('v32.2 independent rebuild', m_new)

actual_diffs = diffs(v30_rows, new_rows)
print('diffs vs v30.1:', actual_diffs)
assert actual_diffs == [14], f'diffs vs v30.1 mismatch: {actual_diffs}'
assert m_new['macro_f1'] > m30['macro_f1'], 'macro did not improve'
assert abs(m_new['mc_macro'] - m30['mc_macro']) < 1e-12, 'MC macro changed; STOP'
assert m_new['per_label']['Unknown']['f1'] >= m30['per_label']['Unknown']['f1'] - 1e-12, 'Unknown F1 dropped; STOP'

if v27_rows is not None:
    d_v27 = diffs(v27_rows, new_rows)
    print('diffs vs v27:', d_v27)
    assert set(d_v27) == EXPECTED_V301_DIFFS_V27 | {14}, f'diffs vs v27 mismatch: {d_v27}'
else:
    d_v27 = None

print(f"Gain vs v30.1 macro-F1: {m_new['macro_f1'] - m30['macro_f1']:+.10f}")


In [ ]:
# ============================================================
# Save artifacts and reload saved preds to verify
# ============================================================
out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
summary = {
    'version': 'v32_2_independent_unparseable_rescue',
    'selection_status': 'SELECT_V32_2_INDEPENDENT',
    'selected_source': 'v30.1 + independent non-MC UNPARSEABLE affirmative-explanation rescue',
    'comparison_to_claude_notebook': claude_inspection,
    'note': 'No v32_1_b_advocate_candidates.json used. Selection is rebuilt from baseline row explanation only.',
    'baseline_v30_1': {k: m30[k] for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
    'metrics': {k: m_new[k] for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},
    'per_label': m_new['per_label'],
    'n_flips_from_v30_1': len(actual_diffs),
    'flipped_indices': actual_diffs,
    'diffs_vs_v27': d_v27,
    'unparseable_audit': audit,
    'risk': {
        'artifact_risk': 'LOW_RELOADED_SAVED_PREDS',
        'overfit_risk': 'MEDIUM_SINGLE_VALIDATION_CASE_BUT_PIPELINE_ARTIFACT',
        'underfit_risk': 'STILL_HIGH_YES_NO_REMAINING',
        'label_collapse': False,
    }
}

for stem in ['v32_2_independent_standard', 'v32_2_independent_full']:
    pred_path = out_dir / f'{stem}_preds.json'
    summ_path = out_dir / f'{stem}_summary.json'
    with open(pred_path, 'w', encoding='utf-8') as f:
        json.dump(new_rows, f, ensure_ascii=False, indent=1)
    with open(summ_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=1)

    # Reload actual saved file and recompute.
    reloaded = json.load(open(pred_path, 'r', encoding='utf-8'))
    rm = compute_metrics(reloaded)
    rd = diffs(v30_rows, reloaded)
    assert rd == [14], f'{stem}: saved diffs mismatch: {rd}'
    assert abs(rm['macro_f1'] - m_new['macro_f1']) < 1e-12, f'{stem}: saved macro mismatch'
    assert abs(rm['mc_macro'] - m30['mc_macro']) < 1e-12, f'{stem}: saved MC changed'
    print('SAVED+VERIFIED:', pred_path)
    print('SAVED+VERIFIED:', summ_path)

# Optional exact compare with Claude v32_2_full_preds.json if present.
existing_v322, existing_path = load_json('v32_2_full_preds.json', required=False)
if existing_v322 is not None:
    same = json.dumps(existing_v322, sort_keys=True, ensure_ascii=False) == json.dumps(new_rows, sort_keys=True, ensure_ascii=False)
    same_preds = [r.get('pred') for r in existing_v322] == [r.get('pred') for r in new_rows]
    print('Existing v32_2_full_preds found:', existing_path)
    print('Exact JSON equal:', same)
    print('Prediction sequence equal:', same_preds)
    assert same_preds, 'Independent predictions do not match existing v32_2 predictions. STOP.'

print('\nALL INDEPENDENT v32.2 REBUILD ASSERTS PASSED')
print(f"NEW BEST (independent rebuild): macro-F1={m_new['macro_f1']:.10f}, acc={m_new['acc']:.10f}")